# backward-fn-signature composite — cx14: back fn replays kwargs via **recipe.kwargs

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `kwargs-pass-through-recipe`, `backward-fn-signature`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "backward-fn-signature"
DD_ATOM_IDS = ["kwargs-pass-through-recipe", "backward-fn-signature"]
DD_SUBTOPICS = ["Backprop: Kwargs pass-through", "Backprop: backward fn signature"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing the canonical back-fn signature with kwargs replay

ARENA back fns share one signature: `back_fn(grad_out, out, *args, **kwargs) -> grad_in`. For an op like `sum(x, dim=1)`, the back fn `sum_back` MUST know which `dim` was reduced — otherwise it can't broadcast the upstream gradient back to the input shape.

The Recipe stores the kwargs the forward call used. The reverse-pass dispatcher splats them in:

```python
grad_in = back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
```

This composite has you write `sum_back` to the canonical signature AND wire it into a dispatcher that replays the kwargs from the Recipe. The point is that the back-fn body NEVER hardcodes `dim` — it reads it from the kwargs the Recipe carried.

### Composite Exercise — back fn replays kwargs via **recipe.kwargs

**Atoms exercised together**: `kwargs-pass-through-recipe`, `backward-fn-signature`

Implement two pieces:

**1. `sum_back(grad_out, out, x, dim=None, keepdim=False)`** — the canonical back fn for `t.sum`. Given `out = x.sum(dim=dim, keepdim=keepdim)`, return `dL/dx` shaped like `x` by broadcasting `grad_out` back over the reduced axis. Two cases:
  - `dim is None` (full reduction): `grad_out` is a 0-D tensor; broadcast it to `x.shape`.
  - `dim is int`: re-insert the reduced axis via `unsqueeze(dim)` (if not `keepdim`), then `expand_as(x)`.

**2. `cx14_dispatch_back(node, grad_out)`** — given a `MiniTensor` `node` whose `recipe` came from a `sum` call, look up the right back fn and call it as `back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)`. Return the resulting grad.

The dispatcher must `**recipe.kwargs`-splat. It must NOT inspect or hardcode any specific kwarg.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

from dataclasses import dataclass, field
from typing import Callable, Optional

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def sum_back(grad_out, out, x, dim=None, keepdim=False):
    """Canonical signature; broadcast grad_out back to x.shape."""
    raise NotImplementedError

# Registry the dispatcher reads from.
BACK_FUNCS = {(t.sum, 0): sum_back}

def cx14_dispatch_back(node, grad_out):
    """Dispatch the back fn for the recipe and replay kwargs via **splat."""
    raise NotImplementedError

def _test_cx14():
    # Build a node as if produced by a wrap_forward_fn(t.sum) call.
    x_raw = t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    out_raw = t.sum(x_raw, dim=1)
    node = MiniTensor(out_raw)
    node.recipe = Recipe(t.sum, (x_raw,), {'dim': 1}, {0: 'fake-parent'})

    # (a) dispatch replays dim=1 → grad spreads back along axis 1
    grad_out = t.tensor([1.0, 1.0])
    g = cx14_dispatch_back(node, grad_out)
    assert g.shape == x_raw.shape, (
        f'sum_back failed to broadcast back to x shape: got {g.shape}'
    )
    assert t.allclose(g, t.ones_like(x_raw)), f'expected ones, got {g}'

    # (b) kwargs is the ONLY source of truth — flip dim=0 and the shape spread flips too.
    out_raw0 = t.sum(x_raw, dim=0)
    node0 = MiniTensor(out_raw0)
    node0.recipe = Recipe(t.sum, (x_raw,), {'dim': 0}, {0: 'fake'})
    g0 = cx14_dispatch_back(node0, t.tensor([1.0, 1.0, 1.0]))
    assert g0.shape == x_raw.shape
    assert t.allclose(g0, t.ones_like(x_raw))

    # (c) keepdim flows through unchanged via the **splat (different inserted shape).
    out_kd = t.sum(x_raw, dim=1, keepdim=True)   # shape (2,1)
    node_kd = MiniTensor(out_kd)
    node_kd.recipe = Recipe(t.sum, (x_raw,), {'dim': 1, 'keepdim': True}, {0: 'fake'})
    g_kd = cx14_dispatch_back(node_kd, t.tensor([[1.0], [1.0]]))
    assert g_kd.shape == x_raw.shape, f'keepdim branch shape: {g_kd.shape}'
    assert t.allclose(g_kd, t.ones_like(x_raw))

    # (d) full reduction (dim=None) — grad_out is 0-D, broadcast to x.shape.
    out_full = t.sum(x_raw)
    node_full = MiniTensor(out_full)
    node_full.recipe = Recipe(t.sum, (x_raw,), {}, {0: 'fake'})
    g_full = cx14_dispatch_back(node_full, t.tensor(1.0))
    assert g_full.shape == x_raw.shape
    assert t.allclose(g_full, t.ones_like(x_raw))

    # (e) the back fn agrees with torch.autograd on a non-unit grad_out.
    x_ref = t.tensor([[2.0, -3.0], [1.0, 4.0]], requires_grad=True)
    y = x_ref.sum(dim=1)
    y.backward(t.tensor([5.0, 7.0]))
    ours = sum_back(t.tensor([5.0, 7.0]), x_ref.detach().sum(dim=1), x_ref.detach(), dim=1)
    assert t.allclose(ours, x_ref.grad), f'disagree with autograd: {ours} vs {x_ref.grad}'
    _dd_passed.add('cx14')

_test_cx14()

<details><summary>Show solution — cx14</summary>

```python
def sum_back(grad_out, out, x, dim=None, keepdim=False):
    # Canonical signature; broadcast grad_out back across the reduced axis.
    if dim is None:
        # Full reduction: grad_out is 0-D; broadcast to x.shape.
        return grad_out.expand(x.shape).clone()
    g = grad_out if keepdim else grad_out.unsqueeze(dim)
    return g.expand_as(x).clone()

BACK_FUNCS = {(t.sum, 0): sum_back}

def cx14_dispatch_back(node, grad_out):
    # Look up by recipe.func; argnum 0 (single-arg op).
    back_fn = BACK_FUNCS[(node.recipe.func, 0)]
    # **splat — kwargs come from the Recipe, never hardcoded.
    return back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
```

The dispatcher line `back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)` is the ONE place kwargs flow from forward into reverse. Storing them on the Recipe (cx13) and splatting them here (cx14) is what keeps `sum_back` op-agnostic — same call shape regardless of op.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx14'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx14',
        'subtopics': ["Backprop: Kwargs pass-through", "Backprop: backward fn signature"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()